In [1]:
# 1. PDF 문서 추출
# uv add langchain-community
# uv add pypdf

from langchain_community.document_loaders import PyPDFLoader

file_path = 'datasets/한글맞춤법 표준어규정 해설.pdf'
loader = PyPDFLoader(file_path)
pages = [] # pdf 내용을 저장할 곳

async for page in loader.alazy_load(): # 한 페이지씩 읽기
    pages.append(page)

/var/folders/r1/x9vbtdd96qv1wnc_0zf8x_y40000gn/T/ipykernel_9464/1387644303.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# 페이지 정보 확인하기
print("페이지 수: ", len(pages))
print("첫 번째 페이지 정보: ", pages[0])

페이지 수:  264
첫 번째 페이지 정보:  page_content='국립국어원 2018-01-08
발간 등록 번호
11-1371028-000712-01
한글 맞춤법
표준어 규정
해설' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2018-12-20T09:36:29+09:00', 'author': 'admin', 'moddate': '2018-12-20T09:38:09+09:00', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'total_pages': 264, 'page': 0, 'page_label': '1'}


In [3]:
# 2. 텍스트 분할하기 (청킹)
# 데이터를 의미 단위로 잘게 나누는 청킹 chunking 작업을 수행.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(pages)



In [4]:
# 청크 정보 확인
print(f"총 {len(docs)}개의 청크 생성 완료")

총 510개의 청크 생성 완료


In [5]:
# 3. 임베딩(단어의 수치화;벡터화) 및 벡터 스토어(DB) 생성 
# 임베딩 모델 : OpenAI의 "text-embedding-3-small" 임베딩 모델
# 벡터 스토어 (벡터 데이터베이스) : chroma
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings # 임베딩 모델
from langchain_chroma import Chroma # 벡터 스토어

DB_PATH = './chroma_db' # 벡터 스토어 경로

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model='text-embedding-3-small'),
    persist_directory=DB_PATH,
    collection_name='korean_pdf' # 문서가 여러개일때는 해당 name 관련된 것만 보는 것이 가능
)

In [11]:
result = vectorstore.similarity_search("구개음화", k=3) # 상위값 3개만
result

[Document(id='3964fd3b-2b8e-4d4b-9341-112b5749f433', metadata={'author': 'admin', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2018-12-20T09:38:09+09:00', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'creationdate': '2018-12-20T09:36:29+09:00', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'page_label': '24', 'page': 23, 'total_pages': 264}, page_content='는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이\n어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 말이므로 구개음화가 실현되지 \n않는다. \n곧이[고지]: 곧-(어근)+-이(부사 파생 접미사)\n곧이어[고디어]: 곧(부사)+이어(부사)\n현재 표준어에서 구개음화는 형태소와 형태소가 결합할 때 일어나는 현상이다. 그러\n므로 ‘마디, 견디다’와 같이 하나의 형태소 내부에서는 구개음화가 일어나지 않는다.'),
 Document(id='0d4adefc-d989-462e-904c-359b5599d301', metadata={'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'moddate': '2018-12-20T09:38:09+09:00', 'page_label': '245', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'author': 'admin

In [9]:
print(len(result))

3


In [10]:
print(result[0].page_content)

는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이
어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 말이므로 구개음화가 실현되지 
않는다. 
곧이[고지]: 곧-(어근)+-이(부사 파생 접미사)
곧이어[고디어]: 곧(부사)+이어(부사)
현재 표준어에서 구개음화는 형태소와 형태소가 결합할 때 일어나는 현상이다. 그러
므로 ‘마디, 견디다’와 같이 하나의 형태소 내부에서는 구개음화가 일어나지 않는다.
